# 03 · Enrutador de proyectos: qué funciona y qué no

**Objetivo:** llega un WhatsApp → decidir a qué proyecto pertenece y en qué carpeta está.

Este notebook está escrito **después** de medirlo, y reescrito dos veces cuando la medición
tumbó lo que daba por hecho. La primera versión acertaba **1 de 8**.

## El resumen, por delante

| tarea | ¿funciona? | medido |
|---|---|---|
| elegir entre proyectos con vocabulario distinto | **sí** | 7/7 |
| clasificar bug / feature / pregunta / admin | **sí** | 6/6 |
| decidir si el mensaje es de trabajo o personal | **no** | 5/10 (azar) |
| `noul` tipo "¿es una tarea?" / "¿es de trabajo?" | **no** | invertido o pegado a 0 |
| detectar "esto no es de ningún proyecto" por umbral | **no** | distribuciones solapadas |

Traducción: Laya **enruta bien entre proyectos**, y **no sabe decirte si el mensaje es de
un proyecto**. Esa puerta hay que ponerla fuera.

## Por qué el umbral no puede detectar "ninguno de estos"

`choice` es un softmax sobre **un conjunto cerrado**. Tiene que repartir toda la masa entre
las etiquetas que le das. Un mensaje fuera de dominio no produce una distribución plana:
produce una distribución **picuda y equivocada**. Medido:

```
"Acuérdate de comprar pan"               -> shop   p1=0.873  margen=0.806
"Se ha caído producción, el dominio..."  -> infra  p1=0.489  margen=0.216
```

El mensaje que no va a ningún proyecto sale **más confiado** que el que sí. No es un bug de
Laya: es una propiedad del softmax de conjunto cerrado. Ningún umbral separa eso.

## 1 · Registro de proyectos

Aquí se concentra casi toda la precisión. Dos reglas, las dos medidas:

**Descripciones = listas de palabras clave. Sin prosa.** Pasar de prosa a keywords subió
6/7 → 7/7 y bajó los tokens de 150 a 81. La prosa ("Tienda online del cliente:",
"Plataforma", "Infraestructura y despliegues") añade masa semántica que **compite** con la
señal. Los tokens que ahorras son presupuesto que recuperas para el mensaje.

**El nombre de la etiqueta también es señal.** Descubierto por accidente al renombrar los
proyectos de ejemplo: con etiquetas opacas (`acme_shop`, `fitleads`) las tres variantes
daban 2/7, 4/7, 6/7; con etiquetas que son palabras con significado (`shop`, `leadgen`)
las mismas variantes dan 4/7, 6/7, 7/7. La clave del dict entra al modelo igual que la
descripción. **Nombra tus proyectos con palabras, no con códigos internos.**

**Sin etiqueta `otros`.** Es un atractor: con `otros` 4/7, sin ella 6/7. Cuesta 2 aciertos.

In [ ]:
# ⚠️ EDITA ESTO con tus proyectos reales. Solo keywords, separadas por comas.
PROYECTOS = {
    "shop": "Shopify, checkout, carrito, pedidos, pagos, tienda",
    "leadgen":  "leads, formularios, CRM, campañas, webhooks",
    "infra":     "servidores, DNS, certificados, Docker, despliegue",
    "laya_lab":  "modelos, clasificador, notebook, embeddings",
}

CARPETAS = {
    "shop": "~/dev/shop",
    "leadgen":  "~/dev/leadgen",
    "infra":     "~/dev/infra",
    "laya_lab":  "~/Documents/workspace/autonomo/pruebas",
}

# Identificadores inequívocos -> decisión cerrada sin modelo.
# No le pidas a un modelo de 421M lo que resuelve un `in`.
PISTAS = {
    "shop": ["shopify", "liquid"],
    "leadgen":  ["leadgen", "lead gen"],
    "infra":     ["docker-compose", "github actions", "nginx"],
    "laya_lab":  ["laya", "huggingface"],
}

assert len(PROYECTOS) <= 10, "11+ opciones = calibración rota en este checkpoint"
print(f"{len(PROYECTOS)} proyectos, calibración válida")

## 2 · La puerta: de dónde viene el mensaje

Como Laya no puede decidir si un mensaje es de trabajo, **lo decide el origen**.
Un WhatsApp siempre trae chat o contacto. Eso es información gratis, determinista y
perfectamente fiable — y le estábamos pidiendo al modelo que la adivinara.

Esta es la lección de ingeniería del notebook: **no modeles lo que ya sabes.**

In [ ]:
# chat/contacto -> ¿va al enrutador?
CHATS_TRABAJO = {"Cliente · soporte", "Equipo dev", "Infra alertas", "Notas"}
CHATS_IGNORAR = {"Familia", "Mamá", "Pádel jueves"}


def pasa_la_puerta(chat):
    if chat in CHATS_TRABAJO:
        return True
    if chat in CHATS_IGNORAR:
        return False
    return None  # desconocido -> preguntar una vez y recordar

## 3 · Las preguntas

Solo las que midieron bien: `proyecto` y `tipo`, ambas `choice` con criteria de keywords.

`urgencia` (`score`) se queda fuera: en la medición se pegó a ~2.0–2.4 en **todos** los
mensajes, incluido "el sábado comemos en casa de mi madre". Es la debilidad declarada del
checkpoint. Si la quieres, úsala para ordenar una cola, nunca para un `if`.

In [ ]:
PREGUNTAS = {
    "proyecto": {
        "type": "choice",
        "instructions": "¿A qué proyecto pertenece este mensaje?",
        "criteria": PROYECTOS,
    },
    "tipo": {
        "type": "choice",
        "instructions": "¿Qué clase de mensaje es?",
        "criteria": {
            "bug": "roto, error, no funciona, caído",
            "feature": "añadir, cambiar, nuevo",
            "pregunta": "duda, dónde, cómo, cuál",
            "admin": "factura, presupuesto, reunión, plazo",
        },
    },
}

## 4 · El prior: así se usan las decisiones anteriores

Laya **no tiene memoria**. Una pasada, sin estado, no aprende en inferencia.

Pero devuelve **probabilidades**, y con eso sí puedes hacer aritmética:

$$\text{score}(p) = P_{\text{laya}}(p)\cdot \text{prior}(p)^{w}$$

`prior` = frecuencia histórica con decaimiento por recencia. `w`: `0` ignora el historial,
`1` es Bayes puro.

⚠️ **Suelo obligatorio en el prior.** Si un proyecto lleva semanas parado y su prior tiende
a 0, un mensaje clarísimo suyo no podrá ganar nunca.

Esta es la razón real para usar Laya en vez de un LLM que devuelve un string:
**no puedes multiplicar un string por un prior.**

In [ ]:
import json
import time
from collections import defaultdict
from pathlib import Path

HISTORIAL = Path("../decisiones.jsonl")  # también es el dataset del fine-tune futuro

VIDA_MEDIA_DIAS = 7.0   # una decisión pesa la mitad cada 7 días
SUELO_PRIOR = 0.05      # ningún proyecto puede morir
PESO_PRIOR = 0.5        # w


def registrar(texto, proyecto, via, ahora=None):
    HISTORIAL.parent.mkdir(parents=True, exist_ok=True)
    with HISTORIAL.open("a", encoding="utf-8") as f:
        f.write(json.dumps({"ts": ahora or time.time(), "texto": texto,
                            "proyecto": proyecto, "via": via}, ensure_ascii=False) + "\n")


def calcular_prior(ahora=None):
    ahora = ahora or time.time()
    pesos = defaultdict(float)
    if HISTORIAL.exists():
        for linea in HISTORIAL.read_text(encoding="utf-8").splitlines():
            if linea.strip():
                d = json.loads(linea)
                pesos[d["proyecto"]] += 0.5 ** (((ahora - d["ts"]) / 86400.0) / VIDA_MEDIA_DIAS)

    total = sum(pesos.values())
    if total == 0:
        return {p: 1.0 / len(PROYECTOS) for p in PROYECTOS}  # sin historial, sin sesgo
    return {p: max(pesos.get(p, 0.0) / total, SUELO_PRIOR) for p in PROYECTOS}


calcular_prior()

## 5 · La decisión

Dos señales de duda, dos fallos **dentro del dominio**:

| señal | significa |
|---|---|
| `p1 < UMBRAL_P` | no reconoce el proyecto |
| `p1 - p2 < UMBRAL_MARGEN` | ambigüedad entre dos proyectos plausibles |

Sirven para mensajes que **sí** son de trabajo. Para los que no, la puerta de la sección 2.

Calculado sobre `probabilities`, no sobre el campo `confidence`: su fórmula no está
documentada y puede cambiar entre versiones.

In [ ]:
from laya import Router

router = Router()

UMBRAL_P = 0.50
UMBRAL_MARGEN = 0.15


def pre_filtro(texto):
    bajo = texto.lower()
    golpes = {p for p, pistas in PISTAS.items() if any(k in bajo for k in pistas)}
    return golpes.pop() if len(golpes) == 1 else None  # 2+ pistas = ambiguo, decide Laya


def decidir(texto, chat=None, usar_prior=True, ahora=None):
    if chat is not None and pasa_la_puerta(chat) is False:
        return {"proyecto": None, "via": "puerta", "p1": 0.0, "margen": 0.0, "extra": None}

    if (directo := pre_filtro(texto)):
        return {"proyecto": directo, "carpeta": CARPETAS.get(directo), "via": "regla",
                "p1": 1.0, "margen": 1.0, "extra": None}

    r = router.predict(texto, PREGUNTAS, model="multilingual", max_len=2048)
    probs = dict(r["answers"]["proyecto"]["probabilities"])

    if usar_prior:
        prior = calcular_prior(ahora)
        ajust = {k: v * prior[k] ** PESO_PRIOR for k, v in probs.items()}
        total = sum(ajust.values())
        probs = {k: v / total for k, v in ajust.items()}

    orden = sorted(probs.items(), key=lambda kv: -kv[1])
    (mejor, p1), (_, p2) = orden[0], orden[1]
    margen = p1 - p2

    if p1 < UMBRAL_P or margen < UMBRAL_MARGEN:
        return {"proyecto": None, "carpeta": None, "via": "ambiguo", "p1": p1,
                "margen": margen, "candidatos": orden[:3], "extra": r["answers"]}

    return {"proyecto": mejor, "carpeta": CARPETAS.get(mejor), "via": "laya",
            "p1": p1, "margen": margen, "extra": r["answers"]}

In [ ]:
import pandas as pd

# ⚠️ Sustituye esto por 20-30 WhatsApps REALES tuyos, de chats de trabajo.
# Estos están limpios y por tanto son optimistas: los de verdad llevan faltas,
# jerga y frases de tres palabras.
BANCO = [
    ("El checkout no deja pagar con tarjeta, los clientes se quejan", "shop"),
    ("¿Puedes añadir un campo de teléfono al formulario de captación?", "leadgen"),
    ("Se ha caído producción, el dominio no resuelve", "infra"),
    ("He probado el clasificador y me da 0.87 de confianza", "laya_lab"),
    ("Hay que renovar el certificado, caduca el viernes", "infra"),
    ("Los pedidos duplicados siguen apareciendo en el panel", "shop"),
    ("¿Te acordaste de mirar lo del webhook?", "leadgen"),
    ("Mándame la factura de septiembre cuando puedas", None),   # ambiguo a propósito
    ("No funciona", None),                                       # sin contexto
]

filas = []
for texto, esperado in BANCO:
    d = decidir(texto, usar_prior=False)   # sin prior: mide el modelo desnudo
    extra = d["extra"] or {}
    filas.append({
        "mensaje": texto[:44], "esperado": esperado or "(ambiguo)",
        "predicho": d["proyecto"] or f"<{d['via']}>",
        "p1": round(d["p1"], 3), "margen": round(d["margen"], 3),
        "tipo": extra["tipo"]["choice"] if extra else None,
    })

df = pd.DataFrame(filas)
claros = df[df["esperado"] != "(ambiguo)"]
print(f"aciertos en casos claros: {(claros['predicho'] == claros['esperado']).sum()}/{len(claros)}")
df

**Qué mirar:**

- **6/7, no 7/7 — y está bien.** La clasificación cruda acierta 7/7 (ablación C, sección 6).
  Aquí sale 6/7 porque `"Se ha caído producción"` da `p1=0.489`, justo por debajo de
  `UMBRAL_P=0.50`, y el pipeline lo manda a preguntar en vez de decidirlo. No es un fallo de
  clasificación: es el umbral funcionando. Enruta 6 solo y escala 1.
- **Los dos `None`** ("factura de septiembre", "No funciona") deberían caer en `<ambiguo>`.
- **`margen` discrimina mejor que `p1`.** Ordena por `margen` y mira si los errores quedan
  abajo. Si sí, ya tienes tu umbral.
- **Preguntar `tipo` en la misma pasada no cuesta precisión** (medido: 7/7 con y sin), pero
  sí tokens: 81 → 144. Gratis en latencia, no en presupuesto de contexto.

## 6 · Las ablaciones (por qué los defaults son estos)

Mismo banco de 7 mensajes, mismo modelo, mismas etiquetas. Solo cambian las descripciones:

| variante | aciertos | tokens |
|---|---|---|
| A · criteria en prosa, **con** `otros` | **4/7** | 178 |
| B · criteria en prosa, sin `otros` | **6/7** | 150 |
| C · criteria de **keywords**, sin `otros` ← el default | **7/7** | 81 |

Dos conclusiones:

1. **`otros` cuesta 2 aciertos.** Canibaliza.
2. **La prosa cuesta 1 acierto más y el doble de tokens.** Keywords > frases.

Y dos cosas que creí y **la medición tumbó**, por si te ahorran el mismo error:

- *"Añadir keywords razonables empeora"* — lo medí con etiquetas opacas (6/7 → 5/7).
  Con etiquetas con significado, añadir 4 keywords no cambia nada (7/7 → 7/7, 81 → 95 tokens).
  El efecto era del nombre de la etiqueta, no de las keywords.
- *"El checkpoint inglés va mejor"* — también artefacto de las etiquetas opacas. Con
  etiquetas con significado el español saca **7/7** y el inglés **6/7**. No fuerces
  `model="english"` esperando ganar precisión; el multilingüe además te da 8192 tokens
  frente a 512.

Ejecuta la celda siguiente para reproducir A/B/C.

In [ ]:
PROSA = {
    "shop": "Tienda online del cliente: Shopify, tema Liquid, checkout, carrito, pedidos, "
                 "devoluciones, pasarela Stripe, stock, repo shop",
    "leadgen": "Plataforma LeadGen: captación de leads, formularios, CRM, campañas, "
                "webhooks entrantes, panel de métricas, repo leadgen",
    "infra": "Infraestructura y despliegues: servidores, Docker, CI, GitHub Actions, "
             "dominios, DNS, certificados SSL, backups, monitorización",
    "laya_lab": "Experimentos con modelos: Laya, clasificadores, notebooks, HuggingFace, "
                "embeddings, evaluación, fine-tuning",
}
OTROS = ("Mensajes personales y recados: citas, familia, compras, quedadas, felicitaciones, "
         "nada relacionado con desarrollo ni clientes")

CLAROS = [(t, e) for t, e in BANCO if e]


def ablacion(nombre, criteria, **kw):
    ok, p1s, toks = 0, [], 0
    for texto, esperado in CLAROS:
        r = router.predict(texto, {"p": {"type": "choice", "criteria": criteria,
                                         "instructions": "¿A qué proyecto pertenece este mensaje?"}}, **kw)
        a = r["answers"]["p"]
        ok += a["choice"] == esperado
        p1s.append(max(a["probabilities"].values()))
        toks = r["usage"]["input_tokens"]
    print(f"{nombre:<44} {ok}/{len(CLAROS)}   tokens={toks:<4} "
          f"p1={min(p1s):.2f}-{max(p1s):.2f}")


ablacion("A · prosa + otros", {**PROSA, "otros": OTROS}, model="multilingual", max_len=4096)
ablacion("B · prosa, sin otros", PROSA, model="multilingual", max_len=4096)
ablacion("C · keywords, sin otros  <- el default", PROYECTOS, model="multilingual", max_len=4096)

### Lo que se midió roto

Tres intentos de detectar "esto no es de ningún proyecto", los tres fallidos:

**`noul` abstractos → invertidos o pegados al suelo.**
```
"¿Este mensaje trata de trabajo...?"   checkout roto -> 0.235   "felicidades" -> 0.854
"¿Este mensaje pide que se haga algo?" todo -> ~0.00-0.15
```
Al revés, y sin rango. Curioso: los `noul` concretos del notebook 01
(*"threaten to cancel"* → 0.879) **sí** funcionan. El patrón es que este checkpoint
discrimina bien por **vocabulario presente en el texto**, y mal cuando la etiqueta exige un
**juicio abstracto** sobre la naturaleza del mensaje.

**`choice` trabajo/personal → 5/10, azar.**
```
"El checkout no deja pagar con tarjeta"   -> personal 0.775   ✗
"Felicidades por el lanzamiento"          -> trabajo  0.879   ✗
```

**Umbral sobre el proyecto → distribuciones solapadas** (el caso del pan, arriba).

Corre la celda siguiente si quieres verlo con tus propios ojos antes de fiarte de mí.

In [ ]:
ROTOS = {
    "es_trabajo": {"type": "noul",
                   "instructions": "¿Este mensaje trata de trabajo, desarrollo de software o clientes?"},
    "es_tarea": {"type": "noul", "instructions": "¿Este mensaje pide que se haga algo?"},
    "ambito": {"type": "choice", "instructions": "¿De qué trata este mensaje?",
               "criteria": {"trabajo": "software, código, clientes, servidores, pedidos, errores, facturas",
                            "personal": "familia, comida, quedada, cumpleaños, médico, compras"}},
}

casos = [("El checkout no deja pagar con tarjeta", "trabajo"),
         ("Se ha caído producción, el dominio no resuelve", "trabajo"),
         ("Mándame el presupuesto del mes que viene", "trabajo"),
         ("Oye que el sábado comemos en casa de mi madre", "personal"),
         ("Felicidades por el lanzamiento, muy buen trabajo", "personal"),
         ("Acuérdate de comprar pan", "personal")]

filas = []
for texto, clase in casos:
    a = router.predict(texto, ROTOS, model="multilingual", max_len=2048)["answers"]
    filas.append({"mensaje": texto[:40], "real": clase,
                  "es_trabajo": round(a["es_trabajo"]["noul"], 3),
                  "es_tarea": round(a["es_tarea"]["noul"], 3),
                  "ambito": a["ambito"]["choice"],
                  "ambito_p": round(max(a["ambito"]["probabilities"].values()), 3)})

pd.DataFrame(filas)

## 7 · El bucle de aprendizaje

Cuando sale `<ambiguo>`, preguntas. La respuesta alimenta el prior desde ya, y será el
dataset del fine-tune cuando juntes 200-300 ejemplos (el repo de laya trae notebook).

Este bucle es el único camino real para que el sistema mejore: Laya no aprende sola.

In [ ]:
def resolver(texto, chat=None, interactivo=False):
    d = decidir(texto, chat=chat)

    if d["proyecto"]:
        registrar(texto, d["proyecto"], d["via"])
        return d
    if d["via"] == "puerta":
        return d

    print(f"No lo tengo claro (p1={d['p1']:.3f}, margen={d['margen']:.3f})")
    for nombre, p in d.get("candidatos", []):
        print(f"   {nombre:<12} {p:.3f}")
    if not interactivo:
        return d

    nombre = input("\n¿Proyecto? (existente o nuevo): ").strip()
    if nombre not in PROYECTOS:
        PROYECTOS[nombre] = input("Keywords (separadas por comas, SIN prosa): ").strip()
        CARPETAS[nombre] = input("Carpeta: ").strip()
        PREGUNTAS["proyecto"]["criteria"] = PROYECTOS
        print(f"Añadido. {len(PROYECTOS)} proyectos"
              + (" — OJO: 11+ rompe la calibración" if len(PROYECTOS) > 10 else ""))

    registrar(texto, nombre, "humano")
    return {"proyecto": nombre, "carpeta": CARPETAS.get(nombre), "via": "humano",
            "p1": d["p1"], "margen": d["margen"]}


resolver("Mándame la factura de septiembre cuando puedas", interactivo=False)

## 8 · Comprobar que el prior mueve algo

Simula una semana trabajando solo en `shop` y relanza un mensaje ambiguo.

In [ ]:
AMBIGUO = "Mándame la factura de septiembre cuando puedas"

sin = decidir(AMBIGUO, usar_prior=False)
print("sin prior :", sin.get("candidatos", [(sin["proyecto"], sin["p1"])])[0])

ahora = time.time()
for i in range(10):
    registrar("sintetico", "shop", "sintetico", ahora=ahora - i * 3600)

print("prior     :", {k: round(v, 3) for k, v in calcular_prior().items()})
con = decidir(AMBIGUO, usar_prior=True)
print("con prior :", con.get("candidatos", [(con["proyecto"], con["p1"])])[0])

In [ ]:
# limpiar el historial sintético
lineas = [l for l in HISTORIAL.read_text(encoding="utf-8").splitlines()
          if l.strip() and json.loads(l).get("via") != "sintetico"]
HISTORIAL.write_text("\n".join(lineas) + ("\n" if lineas else ""), encoding="utf-8")
print(f"{len(lineas)} decisiones reales conservadas")

---

## Conclusión honesta

**Laya sirve para la mitad del problema.** Enruta entre proyectos con vocabulario distinto
(6/7 en español, 7/7 en inglés) en 25 ms, local y gratis. No sabe decidir si un mensaje
pertenece a algún proyecto — ni por etiqueta, ni por umbral, ni por `noul`.

Arquitectura que sí funciona:

```
WhatsApp
   │
   ├─ 1. el CHAT decide si entra          ← determinista, gratis, fiable
   ├─ 2. pre-filtro por identificador     ← un `in`, no un modelo
   ├─ 3. Laya choice (keywords, sin otros)← 25 ms, local
   ├─ 4. prior bayesiano                  ← el historial, en tu código
   └─ 5. margen estrecho -> preguntar / escalar a un LLM
```

**Un LLM (Haiku) haría los pasos 3 y 5 mejor** — razona, entiende nombres propios y contexto
implícito, y no hay que calibrar nada. Razones reales para Laya aquí:

1. **Privacidad.** Son tus WhatsApps. Nada sale de tu máquina. Basta por sí solo.
2. **Probabilidades con las que hacer Bayes.** Un string no se multiplica por un prior.
3. Coste cero, offline, 25 ms.

**Lo que montaría en serio: cascada.** Laya resuelve la mayoría local y gratis; el margen
estrecho escala a un LLM; la respuesta del LLM se registra como dato de entrenamiento.
Es el patrón "model routing" que la propia demo de Laya lista como caso de uso,
aplicado a sí misma.

**Aviso de alcance:** la parte difícil no es este notebook. Es **la ingesta de WhatsApp** —
API Business (aprobación de Meta), `whatsapp-web.js` (no oficial, se rompe con cada
actualización), o un atajo de iOS/Tasker que reenvíe a un endpoint local. Ahí está el 80%
del trabajo real. Escálalo antes de empezar.

### Orden recomendado

1. Mete 20-30 WhatsApps reales en `BANCO` con su etiqueta.
2. Sección 5 **sin prior**. ¿Acierta? Si no, ajusta keywords y repite.
3. Calibra `UMBRAL_P` y `UMBRAL_MARGEN` sobre tus datos.
4. Solo entonces: puerta por chat, prior, persistencia, ingesta.